# SqueezeNet（CIFAR100を用いた画像認識）

---
## 目的
SqueezeNetは，Fire moduleと呼ばれる軽量な畳み込みブロックを用いることで，少ないパラメータ数で高い認識精度を実現する軽量なCNNである．本ノートブックでは，CIFAR100データセットを用いてSqueezeNetをスクラッチから実装し，パラメータ数と計算量の効率化を重視したネットワーク設計について理解する．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
from time import time

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchsummary

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
CIFAR100データセットを読み込みます．CIFAR100はCIFAR10と同じく32×32ピクセルのカラー画像データセットですが，クラス数が100（1クラスあたり学習用500枚，評価用100枚）とより細かいクラス分けになっています．

学習用データに対しては，`RandomCrop`と`RandomHorizontalFlip`によるData Augmentationを適用します．

In [ ]:
transform_train = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                       transforms.RandomHorizontalFlip(),
                                       transforms.ToTensor()])
transform_test = transforms.Compose([transforms.ToTensor()])

train_data = torchvision.datasets.CIFAR100(root="./", train=True, transform=transform_train, download=True)
test_data = torchvision.datasets.CIFAR100(root="./", train=False, transform=transform_test, download=True)

print(len(train_data), len(test_data))

## Fire module
SqueezeNetは，2016年に提案された軽量な畳み込みニューラルネットワークであり，AlexNet相当の認識精度を大幅に少ないパラメータ数で達成することを目的として設計されました．そのアイデアの中心となるのが「Fire module」と呼ばれる畳み込みブロックです．

Fire moduleは，チャンネル数を削減する1×1畳み込み（squeeze）と，それに続いて1×1畳み込みと3×3畳み込みを並列に適用する層（expand）から構成されます．expand層の出力はチャンネル方向に連結（concat）され，次の層に渡されます．

3×3の畳み込みは1×1の畳み込みに比べて9倍のパラメータ数を必要とするため，squeeze層で入力チャンネル数を絞ってからexpand層に渡すことで，ネットワーク全体のパラメータ数を大きく削減しています．

In [ ]:
class Fire(nn.Module):
    def __init__(self, in_channels, squeeze_channels, expand1x1_channels, expand3x3_channels):
        super().__init__()
        self.squeeze = nn.Sequential(
            nn.Conv2d(in_channels, squeeze_channels, kernel_size=1),
            nn.ReLU(inplace=True),
        )
        self.expand1x1 = nn.Sequential(
            nn.Conv2d(squeeze_channels, expand1x1_channels, kernel_size=1),
            nn.ReLU(inplace=True),
        )
        self.expand3x3 = nn.Sequential(
            nn.Conv2d(squeeze_channels, expand3x3_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        x = self.squeeze(x)
        return torch.cat([self.expand1x1(x), self.expand3x3(x)], dim=1)

## SqueezeNetの定義
Fire moduleを積み重ねてSqueezeNet全体を構築します．SqueezeNetは，AlexNetやVGGのような大きな全結合層を持たず，最終的な分類も1×1畳み込みでチャンネル数をクラス数まで削減したのち，Global Average Poolingで空間方向を集約することで行います．これにより，全結合層に起因する多数のパラメータを削減しています．

なお，オリジナルのSqueezeNet（ImageNet，224×224入力）は，7×7畳み込み（stride 2）と3回のMaxPoolingによって特徴マップを大きく縮小しますが，32×32と小さいCIFAR100の入力にこの構成をそのまま適用すると，特徴マップが早い段階で消失してしまいます．そのため，本ノートブックではstemをstride 1の3×3畳み込みに変更し，MaxPoolingの回数も2回に減らしています（詳細は本ノートブック末尾の「ImageNet版SqueezeNetとCIFAR版SqueezeNetの違い」を参照）．

In [ ]:
class SqueezeNet(nn.Module):
    def __init__(self, n_class=100):
        super().__init__()
        # ImageNet版は7x7 stride2の畳み込みと3回のmaxpoolで224x224を13x13まで縮小するが，
        # 32x32入力のCIFARでは縮小しすぎると特徴マップが消失するため，
        # stemをstride1の3x3畳み込みに，maxpoolの回数を2回に減らしている
        self.stem = nn.Sequential(
            nn.Conv2d(3, 96, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

        self.fire2 = Fire(96, 16, 64, 64)
        self.fire3 = Fire(128, 16, 64, 64)
        self.pool1 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)  # 32 -> 16

        self.fire4 = Fire(128, 32, 128, 128)
        self.fire5 = Fire(256, 32, 128, 128)
        self.pool2 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)  # 16 -> 8

        self.fire6 = Fire(256, 48, 192, 192)
        self.fire7 = Fire(384, 48, 192, 192)
        self.fire8 = Fire(384, 64, 256, 256)
        self.fire9 = Fire(512, 64, 256, 256)

        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Conv2d(512, n_class, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.fire2(x)
        x = self.fire3(x)
        x = self.pool1(x)
        x = self.fire4(x)
        x = self.fire5(x)
        x = self.pool2(x)
        x = self.fire6(x)
        x = self.fire7(x)
        x = self.fire8(x)
        x = self.fire9(x)
        x = self.classifier(x)
        return x.view(x.size(0), -1)

## ネットワークの作成
定義したSqueezeNetを作成します．`.to(device)`によりモデルのパラメータを`device`上に配置し，最適化手法にはモーメンタムSGD（学習率0.01，モーメンタム0.9，weight decay 5e-4）を用います．最後に`torchsummary.summary()`でネットワークの詳細情報を表示します．

In [ ]:
model = SqueezeNet(n_class=100)
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

torchsummary.summary(model, (3, 32, 32), device=device.type)

## 学習
読み込んだCIFAR100データセットと作成したネットワークを用いて学習を行います．ミニバッチサイズを128，学習エポック数を20とします．

In [ ]:
batch_size = 128
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)

criterion = nn.CrossEntropyLoss()
criterion = criterion.to(device)

model.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        loss = criterion(y, label)
        model.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    print("epoch: {}, mean loss: {}, mean accuracy: {}, elapsed time: {}".format(
        epoch, sum_loss / len(train_loader), count.item() / len(train_data), time() - train_start))

## テスト
学習したネットワークモデルを用いて評価を行います．

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=100, shuffle=False)

model.eval()

count = 0
with torch.no_grad():
    for image, label in test_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

print("test accuracy: {}".format(count.item() / len(test_data)))

## ImageNet版SqueezeNetとCIFAR版SqueezeNetの違い
このノートブックで実装したSqueezeNetは，CIFAR100（32×32入力）に合わせてダウンサンプリング構成を変更したものです．オリジナルのSqueezeNet（ImageNet, 224×224入力）とは，主に次の点が異なります．

| | ImageNet版 | CIFAR版（本ノートブック） |
|---|---|---|
| stem | 7×7畳み込み，stride 2 | 3×3畳み込み，stride 1 |
| MaxPoolingの回数 | 3回 | 2回 |
| 分類器直前の特徴マップサイズ | 13×13 | 8×8 |
| 出力層 | 1×1畳み込み(512→1000) + Global Average Pooling | 1×1畳み込み(512→100) + Global Average Pooling |

なお，`torchvision.models`にはImageNet版のSqueezeNet（`squeezenet1_0`, `squeezenet1_1`）が実装されており，学習済みモデルも公開されています（[torchvisionのSqueezeNetリファレンス](https://pytorch.org/vision/stable/models.html#id15)）．

## 課題

1. Fire moduleのsqueeze channels（squeeze ratio）を変化させ，パラメータ数と認識精度への影響を確認しましょう．

2. MaxPoolingの配置やダウンサンプリングの回数を変更し（例えば1回だけに減らす，または3回に増やす），特徴マップサイズと認識精度の関係を確認しましょう．

3. 分類器の1×1畳み込み+Global Average Poolingを，Flatten+全結合層に置き換えたネットワークを作成し，パラメータ数と認識精度を比較しましょう．